# Election OCR Pipeline — เขต 6 เชียงใหม่ 2569

รัน top-to-bottom ครั้งเดียว บันทึก checkpoint ทุก unit ถ้า Colab หลุดกลับมารันใหม่ได้เลย

In [1]:
!sudo apt-get install -y poppler-utils -q
!pip install pymupdf typhoon-ocr gdown google-api-python-client -q

Reading package lists...
Building dependency tree...
Reading state information...
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [2]:
from google.colab import drive
drive.mount("/content/drive")

import sys, os, re, time, json, fitz
from itertools import product
from typhoon_ocr import ocr_document
from googleapiclient.discovery import build
from google.colab import auth
from google.auth import default
import gdown

os.environ["TYPHOON_OCR_API_KEY"] = "sk-R9W8qm6m5q1AteUcbC3M2apD8JwlSMPwZBDQRjI9B20f7ukt"

OUTPUT_DIR       = "/content/drive/MyDrive/election_ocr"
LOCAL_BASE       = "/content/survey"
SOURCE_FOLDER_ID = "1Jg1q_2X0STIPwyjRGsogfcYN-Uoy26Z0"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOCAL_BASE, exist_ok=True)
print("ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ready


## 1. Download PDFs

In [3]:
auth.authenticate_user()
creds, _ = default()
service = build("drive", "v3", credentials=creds)

def list_children(pid, mime=None):
    q = f"'{pid}' in parents and trashed=false"
    if mime: q += f" and mimeType='{mime}'"
    res = service.files().list(q=q, fields="files(id, name, mimeType)").execute()
    return sorted(res.get("files", []), key=lambda x: x["name"])

def detect_structure(fid):
    ch = list_children(fid)
    folders = [c for c in ch if c["mimeType"] == "application/vnd.google-apps.folder"]
    files   = [c for c in ch if c["mimeType"] != "application/vnd.google-apps.folder"]
    if any(f["name"].endswith(".pdf") for f in files): return "flat"
    if not folders: return "unknown"
    gc = list_children(folders[0]["id"])
    gf = [c for c in gc if c["mimeType"] == "application/vnd.google-apps.folder"]
    gfi= [c for c in gc if c["mimeType"] != "application/vnd.google-apps.folder"]
    if gfi and not gf: return "tambon_combined"
    if gf and gf[0]["name"].startswith("หน่วย"): return "unit_folder"
    return "registry"

ALL_FILES = []
for amphoe in list_children(SOURCE_FOLDER_ID, mime="application/vnd.google-apps.folder"):
    if "นอกเขต" in amphoe["name"] or "นอกราช" in amphoe["name"]:
        for f in list_children(amphoe["id"]):
            if f["name"].endswith(".pdf"):
                ALL_FILES.append(("_นอกเขต", "-", "-", f["id"], f["name"], "nokhet"))
        continue
    s = detect_structure(amphoe["id"])
    print(f"{amphoe['name']} → {s}")
    if s == "unit_folder":
        for t in list_children(amphoe["id"], mime="application/vnd.google-apps.folder"):
            for u in list_children(t["id"], mime="application/vnd.google-apps.folder"):
                for f in list_children(u["id"]):
                    if f["name"].endswith(".pdf"):
                        ALL_FILES.append((amphoe["name"], t["name"], u["name"], f["id"], f["name"], "บช" if "บช" in f["name"] else "ss"))
    elif s == "tambon_combined":
        for t in list_children(amphoe["id"], mime="application/vnd.google-apps.folder"):
            for f in list_children(t["id"]):
                if f["name"].endswith(".pdf"):
                    ALL_FILES.append((amphoe["name"], t["name"], "combined", f["id"], f["name"], "บช" if "บช" in f["name"] else "combined"))
    elif s == "flat":
        for f in list_children(amphoe["id"]):
            if f["name"].endswith(".pdf"):
                ALL_FILES.append((amphoe["name"], "-", "combined", f["id"], f["name"], "combined"))
    elif s == "registry":
        for reg in list_children(amphoe["id"], mime="application/vnd.google-apps.folder"):
            for t in list_children(reg["id"], mime="application/vnd.google-apps.folder"):
                for f in list_children(t["id"]):
                    if f["name"].endswith(".pdf"):
                        ALL_FILES.append((amphoe["name"], t["name"], "per_unit", f["id"], f["name"], "combined"))

from collections import Counter
print(f"\nรวม {len(ALL_FILES)} PDF")
for a, c in Counter(f[0] for f in ALL_FILES).items(): print(f"  {a}: {c}")

อำเภอพร้าว → tambon_combined
อำเภอเชียงดาว → unit_folder
อำเภอเวียงแหง → flat
อำเภอไชยปราการ → registry

รวม 282 PDF
  _นอกเขต: 8
  อำเภอพร้าว: 22
  อำเภอเชียงดาว: 210
  อำเภอเวียงแหง: 2
  อำเภอไชยปราการ: 40


In [12]:
from googleapiclient.http import MediaIoBaseDownload
import io

def shorten_filename(fname, max_bytes=200):
    encoded = fname.encode("utf-8")
    if len(encoded) <= max_bytes:
        return fname
    name, ext = (fname.rsplit(".", 1) if "." in fname else (fname, ""))
    ext = ("." + ext) if ext else ""
    max_name = max_bytes - len(ext.encode("utf-8"))
    while len(name.encode("utf-8")) > max_name:
        name = name[:-1]
    return name + ext

def download_file(file_id, out_path):
    request = service.files().get_media(fileId=file_id)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    with open(out_path, "wb") as f:
        f.write(fh.getvalue())

for amphoe, tambon, unit, fid, fname, ftype in ALL_FILES:
    d = f"{LOCAL_BASE}/{amphoe}/{tambon}/{unit}"
    os.makedirs(d, exist_ok=True)
    safe_fname = shorten_filename(fname)
    out = f"{d}/{safe_fname}"
    if os.path.exists(out): continue
    print(f"⬇ {amphoe}/{tambon}/{unit}/{safe_fname}")
    try:
        download_file(fid, out)
    except Exception as e:
        print(f"  ❌ {e}")
    time.sleep(1)

print("download done")

⬇ อำเภอพร้าว/ตำบลทุ่งหลวง/combined/รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบบัญชีรายชื่อ (ส.ส.5-18 (บช)) หน่.pdf
⬇ อำเภอพร้าว/ตำบลทุ่งหลวง/combined/รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขตเลือกตั้ง (ส.ส.5-18) หน.pdf
⬇ อำเภอพร้าว/ตำบลน้ำแพร่/combined/รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบบัญชีรายชื่อ (ส.ส.5-18 (บช)) หน่.pdf
⬇ อำเภอพร้าว/ตำบลน้ำแพร่/combined/รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขตเลือกตั้ง (ส.ส.5-18) หน.pdf
⬇ อำเภอพร้าว/ตำบลบ้านโป่ง/combined/รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบบัญชีรายชื่อ (ส.ส.5-18 (บช)) หน่.pdf
⬇ อำเภอพร้าว/ตำบลบ้านโป่ง/combined/รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขตเลือกตั้ง (ส.ส.5-18) หน.pdf
⬇ อำเภอพร้าว/ตำบลป่าตุ้ม/combined/รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบบัญชีรายชื่อ (ส.ส.5-18 (บช)) หน่.pdf
⬇ อำเภอพร้าว/ตำบลป่าตุ้ม/combined/รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขตเลือกตั้ง (ส.ส.5-18) หน.pdf
⬇ อำเภอพร้าว/ตำบลป่าไหน่/combined/รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบบัญชีรายชื่อ (ส.ส.5-18 (

## 2. Core Functions

In [5]:
VOCAB = {"ศูนย์":0,"หนึ่ง":1,"เอ็ด":1,"ยี่":2,"สอง":2,"สาม":3,"สี่":4,"ห้า":5,"หก":6,"เจ็ด":7,"แปด":8,"เก้า":9,"สิบ":10,"ร้อย":100,"พัน":1000}
VOCAB_WORDS = sorted(VOCAB.keys(), key=len, reverse=True)
VC = [set("กถ"),set("ดตฎฏ"),set("นณห"),set("สศษ"),set("ชซฌ"),set("บปข"),set("มท"),set("วย"),set("เแ"),set("าำ"),set("็่้๊๋"),set("17"),set("14"),set("38"),set("60"),set("91"),set("56")]
VCX = VC + [set("คศ"),set("ลส"),set("งจ"),set("ขห"),set("อจ"),set("ูุ"),set("ิี")]

def is_sim(a,b,ex=False): return a==b or any(a in g and b in g for g in (VCX if ex else VC))
def ved(s1,s2,ex=False):
    if s1==s2: return 0
    if len(s1)!=len(s2): return 999
    mm=[(a,b) for a,b in zip(s1,s2) if a!=b]
    return (1 if is_sim(*mm[0],ex=ex) else 999) if len(mm)==1 else 999

def tokenize(text, ex=False):
    text=text.strip().replace("คะแนน","").replace("แนน","").replace(" ","")
    tokens,failed=[],[]
    i=0
    while i<len(text):
        best,bl=None,0
        for w in VOCAB_WORDS:
            chunk=text[i:i+len(w)]
            if len(chunk)<len(w): continue
            d=ved(chunk,w,ex)
            if d<=1 and len(w)>bl: best=(chunk,w,d); bl=len(w)
        if best: tokens.append(best); i+=bl
        else: failed.append(text[i]); i+=1
    return tokens,failed

def to_int(tokens):
    words=[w for _,w,_ in tokens]; r,cur=0,0
    for w in words:
        v=VOCAB[w]
        if v==1000: r+=max(cur,1)*1000; cur=0
        elif v==100: cur=max(cur,1)*100
        elif v==10: m=cur%100 if cur%100!=0 else 1; cur=(cur//100)*100+m*10
        else: cur+=v
    return r+cur

def parse_thai(text):
    t,_=tokenize(text)
    if not t: return None,999,[]
    return to_int(t),max(d for _,_,d in t),t

def word_clean(text):
    t,f=tokenize(text,ex=True)
    if not t or len(f)>1: return False,None
    return True,to_int(t)

def thai2ar(text): return text.translate(str.maketrans("๐๑๒๓๔๕๖๗๘๙","0123456789"))

BD=r"บัตรดี\s+จำนวน\s+(\d+)\s+บัตร\s+\(?([^)\n]+)\)?"
TOT=r"รวมคะแนนทั้งสิ้น[^(\d]*(\d+)\s*\(?([^)\n<]+)\)?"
USED=r"บัตรเลือกตั้งที่ใช้\s+จำนวน\s+(\d+)\s+บัตร\s+\(?([^)\n]+)\)?"
SPOIL=r"บัตรเสีย\s+จำนวน\s+(\d+)\s+บัตร\s+\(?([^)\n]+)\)?"
NOV=r"บัตรที่ไม่เลือกบัญชีรายชื่อ[^(]*(\d+)\s+บัตร\s+\(?([^)\n]+)\)?"
NOV_SS=r"บัตรที่ไม่เลือกผู้สมัครผู้ใด[^(]*(\d+)\s+บัตร\s+\(?([^)\n]+)\)?"
ELIG=r"ผู้มีสิทธิเลือกตั้งตามบัญชีรายชื่อ[^(]*(\d+)\s+คน\s+\(?([^)\n]+)\)?"
TURN=r"ผู้มีสิทธิเลือกตั้งที่มาแสดงตน[^(]*(\d+)\s+คน\s+\(?([^)\n]+)\)?"
ALLOC=r"บัตรเลือกตั้งที่ได้รับจัดสรร\s+จำนวน\s+(\d+)\s+บัตร\s+\(?([^)\n]+)\)?"
REMAIN=r"บัตรเลือกตั้งที่เหลือ\s+จำนวน\s+(\d+)\s+บัตร\s+\(?([^)\n]+)\)?"

def xlast(pat,text):
    ms=list(re.finditer(pat,text,re.DOTALL))
    if not ms: return None,None,999
    m=ms[-1]
    n=int(m.group(1)) if m.group(1).isdigit() else None
    wv,d,_=parse_thai(m.group(2).strip("() "))
    return n,wv,d

def ballot_summary(header):
    def g(pat,t):
        n,w,_=xlast(pat,t)
        if n is not None and w is not None and n==w: return n
        return n if n is not None else w
    v=g(BD,header); sp=g(SPOIL,header); nv=g(NOV,header)
    if nv is None: nv=g(NOV_SS,header)
    u=g(USED,header); al=g(ALLOC,header); rm=g(REMAIN,header)
    sub=alloc="skip"
    if all(x is not None for x in [v,sp,nv]):
        s=v+sp+nv; sub="✅ ok" if u and s==u else f"⚠️ {s}!=used({u})"
    if all(x is not None for x in [u,rm,al]):
        alloc="✅ ok" if u+rm==al else f"⚠️ {u}+{rm}!=alloc({al})"
    return {"eligible":g(ELIG,header),"turnout":g(TURN,header),"ballots_alloc":al,
            "ballots_used":u,"ballots_valid":v,"ballots_spoiled":sp,
            "ballots_no_vote":nv,"ballots_remain":rm,"sub_check":sub,"alloc_check":alloc}

In [6]:
def ocr_retry(pdf_path, page_num, retries=3, wait=30):
    for i in range(retries):
        try: return ocr_document(pdf_or_image_path=pdf_path, page_num=page_num)
        except Exception as e:
            if i<retries-1: print(f"  retry p{page_num}: {e}"); time.sleep(wait)
            else: print(f"  ❌ p{page_num} failed"); return ""

def build_unit_map(pdf_path):
    n=len(fitz.open(pdf_path)); page_types={}
    for p in range(1,n+1):
        text=ocr_retry(pdf_path,p); ta=thai2ar(text)
        has_header = "บัตรดี" in text or "รายงานผลการนับคะแนน" in text
        has_end    = "รวมคะแนนทั้งสิ้น" in text
        has_sig    = "กรรมการประจำหน่วยเลือกตั้ง" in text
        if has_end and has_header:       pt="header_and_end"
        elif has_sig and not has_header: pt="signature"
        elif has_end:                    pt="table_end"
        elif has_header:                 pt="header"
        elif has_sig:                    pt="signature"
        else:                            pt="table_mid"
        page_types[p]=(pt,ta); print(f"    p{p}: {pt}"); time.sleep(4)

    units,cur=[],{}
    for p,(pt,text) in page_types.items():
        if pt=="header":
            if cur and "header_page" in cur and "table_end_page" not in cur:
                cur["table_mid_texts"].append(text); continue
            if cur and "header_page" in cur: units.append(cur)
            cur={"header_page":p,"header_text":text,"table_mid_texts":[]}
        elif pt=="header_and_end":
            if cur and "header_page" in cur: units.append(cur)
            cur={"header_page":p,"header_text":text,"table_mid_texts":[],"table_end_page":p,"table_end_text":text}
            units.append(cur); cur={}
        elif pt=="table_mid" and cur: cur["table_mid_texts"].append(text)
        elif pt=="table_end" and cur: cur["table_end_page"]=p; cur["table_end_text"]=text
        elif pt=="signature" and cur: units.append(cur); cur={}
    if cur and "header_page" in cur: units.append(cur)
    return units

In [7]:
PARTY_NAMES=["ไทยทรัพย์ทวี","เพื่อชาติไทย","ใหม่","มิติใหม่","รวมใจไทย","รวมไทยสร้างชาติ","พลวัต","ประชาธิปไตยใหม่","เพื่อไทย","ทางเลือกใหม่","เศรษฐกิจ","เสรีรวมไทย","รวมพลังประชาชน","ท้องที่ไทย","อนาคตไทย","พลังเพื่อไทย","ไทยชนะ","พลังสังคมใหม่","สังคมประชาธิปไตยไทย","ฟิวชัน","ไทรวมพลัง","ก้าวอิสระ","ปวงชนไทย","วิชชั่นใหม่","เพื่อชีวิตใหม่","คลองไทย","ประชาธิปัตย์","ไทยก้าวหน้า","ไทยภักดี","แรงงานสร้างชาติ","ประชากรไทย","ครูไทยเพื่อประชาชน","ประชาชาติ","สร้างอนาคตไทย","รักชาติ","ไทยพร้อม","ภูมิใจไทย","พลังธรรมใหม่","กรีน","ไทยธรรม","แผ่นดินธรรม","กล้าธรรม","พลังประชารัฐ","โอกาสใหม่","เป็นธรรม","ประชาชน","ประชาไทย","ไทยสร้างไทย","ไทยก้าวใหม่","ประชาอาสาชาติ","พร้อม","เครือข่ายชาวนาแห่งประเทศไทย","ไทยพิทักษ์ธรรม","ความหวังใหม่","ไทยรวมไทย","เพื่อบ้านเมือง","พลังไทยรักชาติ"]

def vote_cell(cell):
    cell=thai2ar(cell.strip())
    if re.match(r"^[-–—]\s*$",cell): return 0,0,True
    m=re.search(r"(\d+)",cell); num=int(m.group(1)) if m else None
    m2=re.search(r"[（(]([^)）]+)[)）]",cell)
    wt=m2.group(1).strip() if m2 else cell
    clean,wv=word_clean(wt)
    return num,wv,clean

def parse_party_table(text):
    rows={}
    for row in re.findall(r"<tr>(.*?)</tr>",text,re.DOTALL):
        cells=re.findall(r"<td[^>]*>(.*?)</td>",row,re.DOTALL)
        cells=[re.sub(r"<[^>]+>","",c).strip() for c in cells]
        if len(cells)<3: continue
        pid_s=thai2ar(cells[0].strip())
        if not re.match(r"^\d+$",pid_s): continue
        pid=int(pid_s)
        if not (1<=pid<=57): continue
        n,w,wc=vote_cell(cells[2])
        rows[pid]={"name":PARTY_NAMES[pid-1],"numeral":n,"word_val":w,"word_clean":wc}
    return rows

def parties_from_unit(unit):
    text = unit.get("header_text","") + "\n" + "\n".join(unit.get("table_mid_texts",[])) + "\n" + unit.get("table_end_text","")
    return parse_party_table(text)

def trusted_total(ht,te):
    bn,bw,bd=xlast(BD,ht); tn,tw,td=xlast(TOT,te)
    cands={bn,bw,tn,tw}-{None}
    def ag(v): return sum([bn==v,bw==v and bd<=1,tn==v,tw==v and td<=1])
    t=next((v for v in cands if ag(v)>=2),None)
    return {"trusted_total":t,"anchor_status":f"✅ {t}" if t else f"⚠️ bd=({bn},{bw}) tot=({tn},{tw})"}

In [8]:
def var(n):
    v={n}; s=str(n)
    for i,ch in enumerate(s):
        for g in [{"1","7"},{"1","4"},{"3","8"},{"6","0"},{"9","1"},{"5","6"}]:
            if ch in g:
                for alt in g:
                    if alt!=ch: v.add(int(s[:i]+alt+s[i+1:]))
    return v

def conf(p, tt=None):
    n,w,wc=p["numeral"],p["word_val"],p["word_clean"]; sc=0.5
    if n is None: sc-=0.3
    if wc and w==n: sc+=0.5
    elif wc and w is not None: sc+=0.1
    else: sc-=0.1
    if n in {1,7,9,3,8}: sc-=0.05
    if n==0 and wc and w==0: return 1.0
    if tt and tt>0 and n is not None:
        r=n/tt
        if r<0.02 and not (wc and w==n): sc-=0.2
        elif r<0.05 and not (wc and w==n): sc-=0.1
    return round(min(max(sc,0.0),1.0),2)

def recover(parties, tt):
    if tt is None: return None,"⚠️ no trusted total",{}
    chosen={pid:(p["numeral"] if p["word_clean"] and p["word_val"]==p["numeral"] else (p["numeral"] or 0)) for pid,p in parties.items()}
    confs={pid:conf(p,tt) for pid,p in parties.items()}
    cur=sum(v for v in chosen.values())
    if cur==tt: return cur,"✅ exact match",chosen
    suspect=sorted([(pid,p) for pid,p in parties.items() if not (p["word_clean"] and p["word_val"]==p["numeral"])],key=lambda x:confs[x[0]])
    def cands(p):
        s=set()
        if p["numeral"] is not None: s.add(p["numeral"]); s.update(var(p["numeral"]))
        if p["word_clean"] and p["word_val"] is not None: s.add(p["word_val"])
        return s
    for pid,p in suspect:
        for c in cands(p):
            if c==chosen[pid]: continue
            if cur-chosen[pid]+c==tt: chosen[pid]=c; return tt,f"✅ single {pid}→{c}",chosen
    pl=sorted([(confs[a]+confs[b],a,pa,b,pb) for i,(a,pa) in enumerate(suspect) for b,pb in suspect[i+1:]],key=lambda x:x[0])
    for _,a,pa,b,pb in pl:
        for ca,cb in product(cands(pa),cands(pb)):
            if ca==chosen[a] and cb==chosen[b]: continue
            if cur-chosen[a]-chosen[b]+ca+cb==tt: chosen[a]=ca;chosen[b]=cb; return tt,f"✅ pair ({a}→{ca},{b}→{cb})",chosen
    tl=sorted([(confs[a]+confs[b]+confs[c],a,pa,b,pb,c,pc) for i,(a,pa) in enumerate(suspect) for j,(b,pb) in enumerate(suspect[i+1:],i+1) for c,pc in suspect[j+1:]],key=lambda x:x[0])
    for _,a,pa,b,pb,c,pc in tl:
        for ca,cb,cc in product(cands(pa),cands(pb),cands(pc)):
            if cur-chosen[a]-chosen[b]-chosen[c]+ca+cb+cc==tt: chosen[a]=ca;chosen[b]=cb;chosen[c]=cc; return tt,f"✅ triple ({a},{b},{c})",chosen
    return cur,f"⚠️ sum={cur} tt={tt} diff={tt-cur}",chosen

In [9]:
def parse_cand_table(text):
    cands,total_row={},None
    for row in re.findall(r"<tr>(.*?)</tr>",text,re.DOTALL):
        cells=re.findall(r"<td[^>]*>(.*?)</td>",row,re.DOTALL)
        cells=[re.sub(r"<[^>]+>","",c).strip() for c in cells]
        if len(cells)<4: continue
        if "รวมคะแนน" in cells[0] or "รวมคะแนน" in " ".join(cells[:3]):
            nc=thai2ar(cells[-2]) if len(cells)>=2 else ""; wc=cells[-1] if cells else ""
            m=re.search(r"(\d+)",nc); numeral=int(m.group(1)) if m else None
            m2=re.search(r"[（(]([^)）]+)[)）]",wc); wt=m2.group(1).strip() if m2 else wc
            cl,wv=word_clean(wt); total_row={"numeral":numeral,"word_val":wv,"word_clean":cl}; continue
        cid_s=thai2ar(cells[0])
        if not re.match(r"^\d+$",cid_s.strip()): continue
        cid=int(cid_s.strip())
        if not (1<=cid<=100): continue
        sc=thai2ar(cells[3]) if len(cells)>3 else ""; wc=cells[4] if len(cells)>4 else ""
        m=re.search(r"(\d+)",sc); numeral=int(m.group(1)) if m else None
        m2=re.search(r"[（(]([^)）]+)[)）]",wc); wt=m2.group(1).strip() if m2 else wc
        cl,wv=word_clean(wt)
        cands[cid]={"name":cells[1] if len(cells)>1 else "","party":cells[2] if len(cells)>2 else "","numeral":numeral,"word_val":wv,"word_clean":cl}
    return cands,total_row

def cands_from_unit(unit):
    text = unit.get("header_text","") + "\n" + "\n".join(unit.get("table_mid_texts",[])) + "\n" + unit.get("table_end_text","")
    return parse_cand_table(text)

def ss_trusted_total(ht, te, tr):
    bn,bw,bd = xlast(BD, ht)
    tn = tr["numeral"]  if tr else None
    tw = tr["word_val"] if tr else None
    td = 0 if (tr and tr["word_clean"]) else 999
    if tn is None:
        # ลองหาจาก table_end ก่อน
        fb,fw,fd = xlast(TOT, te) if te else (None,None,999)
        if fb is not None: tn,tw,td = fb,fw,fd
    if tn is None:
        # fallback: หาจาก header เอง (กรณีนอกเขต ส.ส.)
        fb,fw,fd = xlast(TOT, ht)
        if fb is not None: tn,tw,td = fb,fw,fd
    cands={bn,bw,tn,tw}-{None}
    def ag(v): return sum([bn==v,bw==v and bd<=1,tn==v,tw==v and td==0])
    t=next((v for v in cands if ag(v)>=2),None)
    return {"trusted_total":t,"anchor_status":f"✅ {t}" if t else f"⚠️ bd=({bn},{bw}) tot=({tn},{tw})"}

def ss_trusted_total_nokhet(ht, tr):
    bn, bw, bd = xlast(BD, ht)
    tn = tr["numeral"]  if tr else None
    tw = tr["word_val"] if tr else None
    td = 0 if (tr and tr["word_clean"]) else 999
    cands = {bn, bw, tn, tw} - {None}
    def ag(v): return sum([bn==v, bw==v and bd<=1, tn==v, tw==v and td==0])
    t = next((v for v in cands if ag(v)>=2), None)
    if t is None and bn is not None: t = bn  # fallback บัตรดี numeral
    return {"trusted_total":t, "anchor_status":f"✅ {t}" if t else f"⚠️ bd=({bn},{bw})"}

def conf_ss(p):
    n,w,wc=p["numeral"],p["word_val"],p["word_clean"]; sc=0.5
    if n is None: sc-=0.3
    if wc and w==n: sc+=0.3
    elif wc and w is not None: sc+=0.05
    else: sc-=0.1
    if n in {1,7,9,3,8}: sc-=0.05
    if n==0 and wc and w==0: return 1.0
    return round(min(max(sc,0.0),1.0),2)

def recover_ss(candidates, tt):
    if tt is None: return None,"⚠️ no trusted total",{}
    chosen={cid:(p["numeral"] if p["word_clean"] and p["word_val"]==p["numeral"] else (p["numeral"] or 0)) for cid,p in candidates.items()}
    confs={cid:conf_ss(p) for cid,p in candidates.items()}
    cur=sum(v for v in chosen.values())
    if cur==tt: return cur,"✅ exact match",chosen
    suspect=sorted([(cid,p) for cid,p in candidates.items() if not (p["word_clean"] and p["word_val"]==p["numeral"])],key=lambda x:confs[x[0]])
    def cands(p):
        s=set()
        if p["numeral"] is not None: s.add(p["numeral"]); s.update(var(p["numeral"]))
        if p["word_clean"] and p["word_val"] is not None: s.add(p["word_val"])
        return s
    for cid,p in suspect:
        for c in cands(p):
            if c==chosen[cid]: continue
            if cur-chosen[cid]+c==tt: chosen[cid]=c; return tt,f"✅ single cid={cid}→{c}",chosen
    pl=sorted([(confs[a]+confs[b],a,pa,b,pb) for i,(a,pa) in enumerate(suspect) for b,pb in suspect[i+1:]],key=lambda x:x[0])
    for _,a,pa,b,pb in pl:
        for ca,cb in product(cands(pa),cands(pb)):
            if ca==chosen[a] and cb==chosen[b]: continue
            if cur-chosen[a]-chosen[b]+ca+cb==tt: chosen[a]=ca;chosen[b]=cb; return tt,f"✅ pair ({a}→{ca},{b}→{cb})",chosen
    tl=sorted([(confs[a]+confs[b]+confs[c],a,pa,b,pb,c,pc) for i,(a,pa) in enumerate(suspect) for j,(b,pb) in enumerate(suspect[i+1:],i+1) for c,pc in suspect[j+1:]],key=lambda x:x[0])
    for _,a,pa,b,pb,c,pc in tl:
        for ca,cb,cc in product(cands(pa),cands(pb),cands(pc)):
            if cur-chosen[a]-chosen[b]-chosen[c]+ca+cb+cc==tt: chosen[a]=ca;chosen[b]=cb;chosen[c]=cc; return tt,f"✅ triple ({a},{b},{c})",chosen
    return cur,f"⚠️ sum={cur} tt={tt} diff={tt-cur}",chosen

## 3. Run OCR

In [14]:
def find_pdfs(unit_dir):
    pdfs = [f for f in os.listdir(unit_dir) if f.endswith(".pdf")]
    bch  = [f for f in pdfs if "_0001" in f or "บช" in f]
    ss   = [f for f in pdfs if "หน่วยเลือกตั้งที่" in f or f.startswith("ส.ส.")]
    if bch and ss:   return f"{unit_dir}/{sorted(bch)[0]}", f"{unit_dir}/{sorted(ss)[0]}"
    if pdfs and not bch: c=f"{unit_dir}/{pdfs[0]}"; return c,c
    if bch: return f"{unit_dir}/{bch[0]}", None
    return None, None

def run_amphoe_bch(amphoe_dir, amphoe, out_json):
    res = json.load(open(out_json,encoding="utf-8")) if os.path.exists(out_json) else {}
    for tambon in sorted(os.listdir(amphoe_dir)):
        td = f"{amphoe_dir}/{tambon}"
        if not os.path.isdir(td): continue
        for unit in sorted(os.listdir(td)):
            ud = f"{td}/{unit}"
            if not os.path.isdir(ud): continue
            pdfs = [f for f in os.listdir(ud) if f.endswith(".pdf")]
            bch_files = sorted([f for f in pdfs if "_0001" in f or "บช" in f])
            ss_files  = sorted([f for f in pdfs if "หน่วยเลือกตั้งที่" in f or f.startswith("ส.ส.")])
            res.setdefault(tambon, {})

            if bch_files:
                for bf in bch_files:
                    lbl = re.sub(r'_0001\.pdf$','',bf).strip()
                    lbl = re.sub(r'\.pdf$','',lbl).strip()
                    if lbl in res[tambon]: continue
                    units = build_unit_map(f"{ud}/{bf}")
                    for i, u in enumerate(units):
                        if u.get("table_end_page") == u.get("header_page"): continue  # ข้าม ส.ส.
                        l = lbl if len(units)==1 else f"{lbl}_{i+1}"
                        parts = parties_from_unit(u)
                        tt = trusted_total(u.get("header_text",""), u.get("table_end_text",""))["trusted_total"]
                        fs, status, chosen = recover(parts, tt)
                        bs = ballot_summary(u.get("header_text",""))
                        print(f"    {l}: {status} | parties={len(parts)}")
                        res[tambon][l] = {"trusted_total":tt,"parties_found":len(parts),"final_sum":fs,"status":status,"votes":chosen,"ballot_summary":bs}
                        json.dump(res, open(out_json,"w",encoding="utf-8"), ensure_ascii=False, indent=2)
            else:
                bp, _ = find_pdfs(ud)
                if not bp: print(f"    ⚠️ no pdf"); continue
                exp = [u for u in os.listdir(ud) if not u.startswith(".") and os.path.isdir(f"{ud}/{u}")]
                done = set(res.get(tambon,{}).keys())
                for un in sorted([u for u in exp if u not in done]):
                    units = build_unit_map(f"{ud}/{un}")
                    for i, u in enumerate(units):
                        if u.get("table_end_page") == u.get("header_page"): continue
                        l = un if len(units)==1 else f"{un}_{i+1}"
                        parts = parties_from_unit(u)
                        tt = trusted_total(u.get("header_text",""), u.get("table_end_text",""))["trusted_total"]
                        fs, status, chosen = recover(parts, tt)
                        bs = ballot_summary(u.get("header_text",""))
                        print(f"    {l}: {status} | parties={len(parts)}")
                        res[tambon][l] = {"trusted_total":tt,"parties_found":len(parts),"final_sum":fs,"status":status,"votes":chosen,"ballot_summary":bs}
                        json.dump(res, open(out_json,"w",encoding="utf-8"), ensure_ascii=False, indent=2)
    return res

def run_amphoe_ss(amphoe_dir, amphoe, out_json):
    res = json.load(open(out_json,encoding="utf-8")) if os.path.exists(out_json) else {}
    for tambon in sorted(os.listdir(amphoe_dir)):
        td = f"{amphoe_dir}/{tambon}"
        if not os.path.isdir(td): continue
        for unit in sorted(os.listdir(td)):
            ud = f"{td}/{unit}"
            if not os.path.isdir(ud): continue
            pdfs = [f for f in os.listdir(ud) if f.endswith(".pdf")]
            bch_files = sorted([f for f in pdfs if "_0001" in f or "บช" in f])
            ss_files  = sorted([f for f in pdfs if "หน่วยเลือกตั้งที่" in f or f.startswith("ส.ส.")])
            res.setdefault(tambon, {})

            if ss_files:
                for sf in ss_files:
                    lbl = re.sub(r'\.pdf$','',sf).strip()
                    if lbl in res[tambon]: continue
                    units = build_unit_map(f"{ud}/{sf}")
                    for i, u in enumerate(units):
                        if u.get("table_end_page") != u.get("header_page") and u.get("table_end_page") is not None: continue
                        l = lbl if len(units)==1 else f"{lbl}_{i+1}"
                        cands, tr = cands_from_unit(u)
                        ti = ss_trusted_total(u.get("header_text",""), u.get("table_end_text",""), tr)
                        fs, status, chosen = recover_ss(cands, ti["trusted_total"])
                        bs = ballot_summary(u.get("header_text",""))
                        cinfo = {cid:{"name":p["name"],"party":p["party"]} for cid,p in cands.items()}
                        print(f"    {l}: {status} | cands={len(cands)}")
                        res[tambon][l] = {"trusted_total":ti["trusted_total"],"anchor_status":ti["anchor_status"],"candidates_found":len(cands),"final_sum":fs,"status":status,"votes":chosen,"candidate_info":cinfo,"ballot_summary":bs}
                        json.dump(res, open(out_json,"w",encoding="utf-8"), ensure_ascii=False, indent=2)
            elif bch_files:
                # combined file — ดึง ส.ส. unit จาก บช file
                for bf in bch_files:
                    lbl = re.sub(r'_0001\.pdf$','',bf).strip()
                    lbl = re.sub(r'\.pdf$','',lbl).strip()
                    if lbl in res[tambon]: continue
                    units = build_unit_map(f"{ud}/{bf}")
                    for i, u in enumerate(units):
                        if u.get("table_end_page") != u.get("header_page"): continue  # เอาเฉพาะ ส.ส.
                        l = lbl if len(units)==1 else f"{lbl}_{i+1}"
                        cands, tr = cands_from_unit(u)
                        ti = ss_trusted_total(u.get("header_text",""), u.get("table_end_text",""), tr)
                        fs, status, chosen = recover_ss(cands, ti["trusted_total"])
                        bs = ballot_summary(u.get("header_text",""))
                        cinfo = {cid:{"name":p["name"],"party":p["party"]} for cid,p in cands.items()}
                        print(f"    {l}: {status} | cands={len(cands)}")
                        res[tambon][l] = {"trusted_total":ti["trusted_total"],"anchor_status":ti["anchor_status"],"candidates_found":len(cands),"final_sum":fs,"status":status,"votes":chosen,"candidate_info":cinfo,"ballot_summary":bs}
                        json.dump(res, open(out_json,"w",encoding="utf-8"), ensure_ascii=False, indent=2)
            else:
                _, sp = find_pdfs(ud)
                if not sp: continue
                units = build_unit_map(sp)
                for i, u in enumerate(units):
                    lbl = unit if len(units)==1 else f"{unit}_{i+1}"
                    if lbl in res.get(tambon,{}): continue
                    cands, tr = cands_from_unit(u)
                    ti = ss_trusted_total(u.get("header_text",""), u.get("table_end_text",""), tr)
                    fs, status, chosen = recover_ss(cands, ti["trusted_total"])
                    bs = ballot_summary(u.get("header_text",""))
                    cinfo = {cid:{"name":p["name"],"party":p["party"]} for cid,p in cands.items()}
                    print(f"    {lbl}: {status} | cands={len(cands)}")
                    res[tambon][lbl] = {"trusted_total":ti["trusted_total"],"anchor_status":ti["anchor_status"],"candidates_found":len(cands),"final_sum":fs,"status":status,"votes":chosen,"candidate_info":cinfo,"ballot_summary":bs}
                    json.dump(res, open(out_json,"w",encoding="utf-8"), ensure_ascii=False, indent=2)
    return res

In [16]:
t0=time.time()

for amphoe in os.listdir(LOCAL_BASE):
    ad = f"{LOCAL_BASE}/{amphoe}"
    print("=" * 50)
    print(amphoe)

    bj=f"{OUTPUT_DIR}/{amphoe}_confidence.json"
    sj=f"{OUTPUT_DIR}/{amphoe}_ss.json"

    rb=run_amphoe_bch(ad,amphoe,bj)
    rs=run_amphoe_ss(ad,amphoe,sj)

    for res,suf in [(rb,"_good.json"),(rs,"_ss_good.json")]:
        good={}
        for tambon,units in res.items():
            for lbl,r in units.items():
                if "exact match" in r["status"] or "recovered" in r["status"]:
                    good[f"{tambon}/{lbl}"]=r
        json.dump(good,open(f"{OUTPUT_DIR}/{amphoe}{suf}","w",encoding="utf-8"),ensure_ascii=False,indent=2)

    tot=sum(len(v) for v in rb.values())
    ex=sum(1 for t in rb.values() for r in t.values() if "exact match" in r["status"])
    rc=sum(1 for t in rb.values() for r in t.values() if "recovered" in r["status"])
    print(f"  บช: {tot} | exact={ex} recovered={rc} failed={tot-ex-rc}")

print(f"✅ done in {(time.time()-t0)/3600:.1f}h")

อำเภอพร้าว
    p1: header
    p2: table_mid
    p3: table_end
    p4: signature
    p5: header
    p6: table_mid
    p7: table_end
    p8: signature
    p9: header
    p10: table_mid
    p11: table_end
    p12: signature
    p13: header
    p14: table_mid
    p15: table_end
    p16: signature
    p17: header
    p18: table_mid
    p19: table_end
    p20: signature
    p21: header
    p22: table_mid
    p23: table_end
    p24: signature
    รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบบัญชีรายชื่อ (ส.ส.5-18 (บช)) หน่_1: ✅ single 12→6 | parties=57
    รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบบัญชีรายชื่อ (ส.ส.5-18 (บช)) หน่_2: ✅ triple (8,11,9) | parties=57
    รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบบัญชีรายชื่อ (ส.ส.5-18 (บช)) หน่_3: ✅ single 42→0 | parties=57
    รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบบัญชีรายชื่อ (ส.ส.5-18 (บช)) หน่_4: ✅ exact match | parties=57
    รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบบัญชีรายชื่อ (ส.ส.5-18 (บช)) หน่_5: ✅ exact match | parties=57
    รายงานผ

KeyboardInterrupt: 

## 4. Summary

In [ ]:
import glob
print("\n📊 OCR Summary")
print(f"{'file':<40} {'units':>6} {'good':>6} {'pct':>6}")
print("-"*55)
for fpath in sorted(glob.glob(f"{OUTPUT_DIR}/*_confidence.json")):
    with open(fpath,encoding="utf-8") as f: d=json.load(f)
    tot=sum(len(v) for v in d.values())
    good=sum(1 for t in d.values() for r in t.values() if "exact match" in r["status"] or "recovered" in r["status"])
    name=os.path.basename(fpath).replace("_confidence.json","")
    print(f"  {name:<38} {tot:>6} {good:>6} {100*good//tot if tot else 0:>5}%")
print("\n📁 Files saved to:", OUTPUT_DIR)